In [7]:
import random

# Task 1: data setup
items = ['N1', 'N2', 'N3', 'N4', 'N5', 'N6']
values = [14, 23, 8, 9, 17, 15]
weights = [1, 3, 7, 4, 5, 6]
MAX_WEIGHT = 10  # 10 kg limit

# GA parameters
POP_SIZE = 10
MAX_GEN = 50
MUT_PROB = 0.1

def generate_individual(n):
    """Create a random binary chromosome of length n."""
    return [random.choice([0, 1]) for _ in range(n)]

def fitness(individual):
    """Compute total value and return 0 if the weight limit is exceeded."""
    total_weight = sum(individual[i] * weights[i] for i in range(len(individual)))
    total_value = sum(individual[i] * values[i] for i in range(len(individual)))
    
    # Strong penalty for overweight solutions
    if total_weight > MAX_WEIGHT:
        return 0
    return total_value

def roulette_wheel_selection(population, fitnesses):
    """Select one individual with probability proportional to fitness."""
    total_fit = sum(fitnesses)
    if total_fit == 0:
        return random.choice(population)
    
    pick = random.uniform(0, total_fit)
    current = 0
    for i, ind in enumerate(population):
        current += fitnesses[i]
        if current > pick:
            return ind
    return population[-1]

def crossover(parent1, parent2):
    """Create two offspring using one-point crossover."""
    pt = random.randint(1, len(parent1) - 1)
    offspring1 = parent1[:pt] + parent2[pt:]
    offspring2 = parent2[:pt] + parent1[pt:]
    return offspring1, offspring2

def mutate(individual, prob):
    """Flip each bit with probability `prob`."""
    return [1 - bit if random.random() < prob else bit for bit in individual]

def solve_knapsack_ga():
    n = len(items)
    population = [generate_individual(n) for _ in range(POP_SIZE)]
    
    best_overall = None
    best_fitness = -1
    
    # Run evolution for a fixed number of generations
    for _ in range(MAX_GEN):
        fitnesses = [fitness(ind) for ind in population]
        
        # Track the best solution seen so far
        for i, fit in enumerate(fitnesses):
            if fit > best_fitness:
                best_fitness = fit
                best_overall = population[i]
                
        new_population = []
        # Build the next generation
        while len(new_population) < POP_SIZE:
            p1 = roulette_wheel_selection(population, fitnesses)
            p2 = roulette_wheel_selection(population, fitnesses)
            o1, o2 = crossover(p1, p2)
            new_population.extend([mutate(o1, MUT_PROB), mutate(o2, MUT_PROB)])
            
        # Replace the old population with the new one
        population = new_population[:POP_SIZE]

    # Decode and print the best solution
    selected_items = [items[i] for i in range(n) if best_overall[i] == 1]
    total_w = sum(weights[i] for i in range(n) if best_overall[i] == 1)
    
    print(f"Best solution found: {best_overall}")
    print(f"Selected items: {selected_items}")
    print(f"Total value: {best_fitness}, total weight: {total_w} kg")

# Run Task 1
solve_knapsack_ga()


Best solution found: [1, 1, 0, 0, 1, 0]
Selected items: ['N1', 'N2', 'N5']
Total value: 54, total weight: 9 kg


In [8]:
import time
from collections import deque


class CSPSolver:
    def __init__(self):
        """Initialize the CSP."""
        self.variables = []
        self.domains = {}
        self.constraints = {}
        self.backtracks = 0

    def add_variable(self, var, domain):
        """Add a variable with its domain."""
        self.variables.append(var)
        self.domains[var] = list(domain)

    def add_constraint(self, var1, var2, constraint_fn):
        """Add a binary constraint between var1 and var2."""
        self.constraints[(var1, var2)] = constraint_fn
        self.constraints[(var2, var1)] = lambda v2, v1: constraint_fn(v1, v2)

    def is_consistent(self, var, value, assignment):
        """Return True if assigning value to var does not violate any constraint."""
        for assigned_var, assigned_value in assignment.items():
            if (var, assigned_var) in self.constraints:
                if not self.constraints[(var, assigned_var)](value, assigned_value):
                    return False
        return True

    def degree(self, var, assignment):
        """Count unassigned neighbors (Degree heuristic)."""
        count = 0
        for v in self.variables:
            if v != var and v not in assignment and (var, v) in self.constraints:
                count += 1
        return count

    def select_unassigned_variable(self, assignment, use_heuristics=False):
        """Return the next unassigned variable."""
        unassigned = [v for v in self.variables if v not in assignment]

        if not use_heuristics:
            return unassigned[0]

        # MRV, then Degree as tie-breaker
        return min(unassigned, key=lambda v: (len(self.domains[v]), -self.degree(v, assignment)))

    def order_domain_values(self, var, assignment):
        """Return domain values for var."""
        return self.domains[var]

    def revise(self, xi, xj):
        """Remove values from domains[xi] that have no support in domains[xj]."""
        revised = False
        constraint = self.constraints.get((xi, xj))
        if not constraint:
            return False

        for x in list(self.domains[xi]):
            if not any(constraint(x, y) for y in self.domains[xj]):
                self.domains[xi].remove(x)
                revised = True
        return revised

    def ac3(self):
        """Enforce arc consistency using AC-3."""
        queue = deque(self.constraints.keys())

        while queue:
            xi, xj = queue.popleft()
            if self.revise(xi, xj):
                if not self.domains[xi]:
                    return False
                for xk in self.variables:
                    if xk != xi and xk != xj and (xk, xi) in self.constraints:
                        queue.append((xk, xi))
        return True

    def backtrack(self, assignment, use_fc=False, use_heuristics=False):
        """Core recursive backtracking search."""
        if len(assignment) == len(self.variables):
            return assignment

        var = self.select_unassigned_variable(assignment, use_heuristics)

        for value in self.order_domain_values(var, assignment):
            if self.is_consistent(var, value, assignment):
                assignment[var] = value

                pruned_values = []
                failure = False

                if use_fc:
                    # Forward Checking
                    for neighbor in self.variables:
                        if neighbor not in assignment and (var, neighbor) in self.constraints:
                            constraint = self.constraints[(var, neighbor)]
                            for neighbor_val in list(self.domains[neighbor]):
                                if not constraint(value, neighbor_val):
                                    self.domains[neighbor].remove(neighbor_val)
                                    pruned_values.append((neighbor, neighbor_val))

                            if not self.domains[neighbor]:
                                failure = True
                                break

                if not failure:
                    result = self.backtrack(assignment, use_fc, use_heuristics)
                    if result is not None:
                        return result

                del assignment[var]
                self.backtracks += 1

                for n, val in pruned_values:
                    self.domains[n].append(val)

        return None

    def solve(self, use_fc=False, use_heuristics=False, run_ac3=False):
        """Solve the CSP and report metrics."""
        self.backtracks = 0

        if run_ac3:
            is_consistent = self.ac3()
            print("\nAC-3 complete.")
            print("Reduced domains:")
            for v in self.variables:
                print(f"{v} : {self.domains[v]}")

            if not is_consistent:
                print("CSP is unsolvable according to AC-3.")
                return None

        start_time = time.time()
        result = self.backtrack({}, use_fc, use_heuristics)
        end_time = time.time()

        mode_str = "Backtracking"
        if use_fc:
            mode_str += " + Forward Checking"
        if use_heuristics:
            mode_str += " + MRV + Degree"

        print(f"Mode: {mode_str}")
        print(f"Solution: {result}")
        print(f"Backtracks: {self.backtracks}")
        print(f"Time: {end_time - start_time:.4f}s\n")

        return result, self.backtracks, (end_time - start_time)


In [9]:
def setup_map_coloring():
    solver = CSPSolver()
    variables = ['WA', 'NT', 'SA', 'Q', 'NSW', 'V', 'T']
    domains = ['R', 'G', 'B']

    for v in variables:
        solver.add_variable(v, domains)

    # Adjacent regions must have different colors
    edges = [
        ('WA', 'NT'), ('WA', 'SA'), ('NT', 'SA'), ('NT', 'Q'),
        ('SA', 'Q'), ('SA', 'NSW'), ('SA', 'V'), ('Q', 'NSW'), ('NSW', 'V')
    ]

    for v1, v2 in edges:
        solver.add_constraint(v1, v2, lambda x, y: x != y)

    return solver


def setup_n_queens(n=4):
    solver = CSPSolver()
    variables = [f"Q{i}" for i in range(1, n + 1)]
    domains = list(range(1, n + 1))

    for v in variables:
        solver.add_variable(v, domains)

    # Constraints: different rows and not on same diagonal
    for i in range(1, n + 1):
        for j in range(i + 1, n + 1):
            def make_constraint(i_idx, j_idx):
                return lambda x, y: x != y and abs(x - y) != abs(i_idx - j_idx)
            solver.add_constraint(f"Q{i}", f"Q{j}", make_constraint(i, j))

    return solver


def run_comparison(problem_name, setup_func):
    modes = [
        {"name": "Backtracking only", "fc": False, "heu": False},
        {"name": "Backtracking + Forward Checking", "fc": True, "heu": False},
        {"name": "Backtracking + FC + MRV + Degree", "fc": True, "heu": True},
    ]

    results = []
    print(f"Solving: {problem_name}")

    # AC-3 demonstration
    solver_ac3 = setup_func()
    solver_ac3.solve(run_ac3=True)

    for mode in modes:
        solver = setup_func()  # fresh solver each run
        _, backtracks, t_elapsed = solver.solve(
            use_fc=mode["fc"], use_heuristics=mode["heu"]
        )
        results.append((mode["name"], backtracks, t_elapsed))

    print(f"Problem: {problem_name}")
    print(f"{'Mode':<40} | {'Backtracks':<10} | {'Time'}")
    print("-" * 70)
    for name, bts, t in results:
        print(f"{name:<40} | {bts:<10} | {t * 1000:.2f}ms")
    print("\n" + "=" * 70 + "\n")


# Direct execution in notebook cell
run_comparison("Australia Map Coloring", setup_map_coloring)
run_comparison("4-Queens", lambda: setup_n_queens(4))

Solving: Australia Map Coloring

AC-3 complete.
Reduced domains:
WA : ['R', 'G', 'B']
NT : ['R', 'G', 'B']
SA : ['R', 'G', 'B']
Q : ['R', 'G', 'B']
NSW : ['R', 'G', 'B']
V : ['R', 'G', 'B']
T : ['R', 'G', 'B']
Mode: Backtracking
Solution: {'WA': 'R', 'NT': 'G', 'SA': 'B', 'Q': 'R', 'NSW': 'G', 'V': 'R', 'T': 'R'}
Backtracks: 0
Time: 0.0000s

Mode: Backtracking
Solution: {'WA': 'R', 'NT': 'G', 'SA': 'B', 'Q': 'R', 'NSW': 'G', 'V': 'R', 'T': 'R'}
Backtracks: 0
Time: 0.0000s

Mode: Backtracking + Forward Checking
Solution: {'WA': 'R', 'NT': 'G', 'SA': 'B', 'Q': 'R', 'NSW': 'G', 'V': 'R', 'T': 'R'}
Backtracks: 0
Time: 0.0000s

Mode: Backtracking + Forward Checking + MRV + Degree
Solution: {'SA': 'R', 'NT': 'G', 'Q': 'B', 'NSW': 'G', 'WA': 'B', 'V': 'B', 'T': 'R'}
Backtracks: 0
Time: 0.0001s

Problem: Australia Map Coloring
Mode                                     | Backtracks | Time
----------------------------------------------------------------------
Backtracking only                    